# GazeNet Training Pipeline

This notebook covers the training of **GazeNet**, which is responsible for detecting **Dyslexia** (regressions/fixations) and **ADHD** (gaze tracking off-task indicators) from eye-movement and tracking coordinate sequences.

## Modality & Scope
- Input: Eye gaze point coordinate time-series sequence `[x, y, timestamp]` and extracted fixation features
- Output: Binary risk classification (0 = Typical reader, 1 = Dyslexia risk indicator)
- Base Network: **LSTM sequence classifier**

## Target Runtime
- Google Colab (Free T4 GPU)

## Step 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q tensorflow pandas numpy scikit-learn matplotlib zenodo-get

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## Step 2: Dataset Acquisition
We use the **ETDD70 (Eye-Tracking Dyslexia Dataset)** from Zenodo (`13332134`). This dataset contains gaze recordings from 70 children (normal vs. dyslexia).

In [ ]:
os.makedirs('dataset/etdd70', exist_ok=True)
os.makedirs('output', exist_ok=True)

# Download via zenodo_get tool
!zenodo_get 13332134 -o dataset/etdd70

print("ETDD70 Dataset successfully fetched.")

## Step 3: Feature Engineering & Preprocessing
We process gaze log entries. For an LSTM sequence processor, we slice fixed-length sequences (e.g. 100 timesteps) of gaze positions.

In [ ]:
def load_gaze_sequences(data_dir, seq_len=100):
    sequences = []
    labels = []
    
    csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    
    if len(csv_files) == 0:
        print("No CSV files found in dataset path. Simulating dummy gaze dataset for notebook build validation...")
        for i in range(200):
            # 100 timesteps, 6 features: [x, y, dx, dy, speed, is_fixation]
            seq = np.random.randn(seq_len, 6)
            sequences.append(seq)
            labels.append(1 if i % 2 == 0 else 0)
        return np.array(sequences), np.array(labels)
        
    for file in csv_files:
        filepath = os.path.join(data_dir, file)
        df = pd.read_csv(filepath)
        
        if len(df) >= seq_len:
            coords = df[['x', 'y']].values[:seq_len]
            diffs = np.diff(coords, axis=0, prepend=coords[0:1])
            speed = np.linalg.norm(diffs, axis=1, keepdims=True)
            features = np.hstack([coords, diffs, speed, np.zeros((seq_len, 1))])
            sequences.append(features)
            is_dyslexia = 1 if 'dys' in file.lower() else 0
            labels.append(is_dyslexia)
            
    return np.array(sequences), np.array(labels)

sequences, labels = load_gaze_sequences('dataset/etdd70', seq_len=100)
print(f"Loaded shape: sequences={sequences.shape}, labels={labels.shape}")

## Step 4: Model Architecture & Training
We build a 2-layer LSTM network with recurrent dropout to handle temporal dependencies.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(sequences, labels, test_size=0.2, random_state=42)

timesteps = sequences.shape[1]
features_dim = sequences.shape[2]

model = tf.keras.Sequential([
    layers.Input(shape=(timesteps, features_dim)),
    layers.LSTM(128, return_sequences=True, recurrent_dropout=0.2),
    layers.LSTM(64, recurrent_dropout=0.2),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Binary risk probability
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC()]
)

print(model.summary())

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=16
)

## Step 5: Export to TFLite (Float16 Quantized)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()
output_path = 'output/seren_gazenet.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model successfully exported to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")

## Step 6: Automated Behavioral Validation Check

In [ ]:
print("\n--- Running Behavioral Validation Check ---")
interpreter = tf.lite.Interpreter(model_path=output_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

test_inputs = [
    np.random.normal(0.0, 1.0, (1, 100, 6)).astype(np.float32),
    np.random.normal(0.5, 0.5, (1, 100, 6)).astype(np.float32),
    np.random.normal(-0.5, 0.2, (1, 100, 6)).astype(np.float32)
]

outputs = []
for inp in test_inputs:
    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]['index'])
    outputs.append(out.flatten())
    print(f"Gaze Input -> Output risk probability: {out.flatten()}")

outputs = np.array(outputs)
std_devs = np.std(outputs, axis=0)
max_std = np.max(std_devs)
print(f"Max standard deviation across outputs: {max_std:.4f}")
assert max_std > 0.005, "FAILED: GazeNet outputs are identical! The weights did not train or converge."
print("PASSED: GazeNet behavioral check succeeded.")